Quote from GLoVe github under [`scr/README.md`](https://github.com/stanfordnlp/GloVe/tree/master/src)

   > To train your own GloVe vectors, first you'll need to prepare your corpus as a single text file with all words separated by one or more spaces or tabs. If your corpus has multiple documents, the documents (only) should be separated by new line characters. Cooccurrence contexts for words do not extend past newline characters. Once you create your corpus, you can train GloVe vectors using the following 4 tools. An example is included in demo.sh, which you can modify as necessary.

# Importing the Data

In [1]:
import pandas as pd
import numpy as np

In [2]:
import os
print(os.getcwd())

/Users/caden/st_david-s-beacon/website/scripts/word embeddings


In [3]:
psalms_verses = pd.read_csv("../../data/csv/cleaned_psalm_verses.csv")
psalms_verses

,tradition,text,psalm_num,verse_num,verse
0,Orthodox,Bible,1,1,Blessed is the man Who walks not in the counse...
1,Orthodox,Bible,1,2,But his will is in the law of the Lord And in ...
2,Orthodox,Bible,1,3,He shall be like a tree Planted by streams of ...
3,Orthodox,Bible,1,4,Not so are the ungodly not so But they are lik...
4,Orthodox,Bible,1,5,Therefore the ungodly shall not rise in the ju...
...,...,...,...,...,...
5000,Orthodox,Psalter,150,64,"Butter of kine, and milk of sheep, with fat of..."
5001,Orthodox,Psalter,150,65,"So Jacob ate, and was filled; and the beloved ..."
5002,Orthodox,Psalter,150,66,"They provoked Me to anger with strange gods, a..."
5003,Orthodox,Psalter,150,67,"They sacrificed unto demons, not to God; to go..."


In [4]:
# Grouped Psalms (Bible & Psalter)
psalms = pd.read_csv("../../data/csv/grouped_psalm.csv")
psalms

,Unnamed: 0,tradition,text,psalm_num,verse,cleaned_verse
0,0,Orthodox,Bible,1,Blessed is the man Who walks not in the counse...,blessed man walk counsel ungodly stand way sin...
1,1,Orthodox,Bible,2,Why do the nations rage And the people meditat...,nation rage people meditate vain thing king ea...
2,2,Orthodox,Bible,3,A psalm by David when he fled from the face of...,psalm david fled face son absalom olord afflic...
3,3,Orthodox,Bible,4,For the End in psalms an ode by David You hear...,end psalm ode david heard icalled god righteou...
4,4,Orthodox,Bible,5,For the End concerning the inheritance a psalm...,end concerning inheritance psalm david give ea...
...,...,...,...,...,...,...
296,296,Orthodox,Psalter,146,The Lord doth build up Jerusalem; He shall gat...,lord doth build jerusalem ; shall gather toget...
297,297,Orthodox,Psalter,147,"Praise the Lord, O Jerusalem; praise thy God, ...","praise lord , jerusalem ; praise thy god , zio..."
298,298,Orthodox,Psalter,148,Praise ye the Lord from the heavens; praise Hi...,praise ye lord heaven ; praise highest . prais...
299,299,Orthodox,Psalter,149,"Sing unto the Lord a new song, His praise is i...","sing unto lord new song , praise congregation ..."


In [5]:
# Renaming the last two columns as it should be psalm
psalms = psalms.rename(columns={'verse':'psalm',"cleaned_verse": "cleaned_psalm"})
psalms

,Unnamed: 0,tradition,text,psalm_num,psalm,cleaned_psalm
0,0,Orthodox,Bible,1,Blessed is the man Who walks not in the counse...,blessed man walk counsel ungodly stand way sin...
1,1,Orthodox,Bible,2,Why do the nations rage And the people meditat...,nation rage people meditate vain thing king ea...
2,2,Orthodox,Bible,3,A psalm by David when he fled from the face of...,psalm david fled face son absalom olord afflic...
3,3,Orthodox,Bible,4,For the End in psalms an ode by David You hear...,end psalm ode david heard icalled god righteou...
4,4,Orthodox,Bible,5,For the End concerning the inheritance a psalm...,end concerning inheritance psalm david give ea...
...,...,...,...,...,...,...
296,296,Orthodox,Psalter,146,The Lord doth build up Jerusalem; He shall gat...,lord doth build jerusalem ; shall gather toget...
297,297,Orthodox,Psalter,147,"Praise the Lord, O Jerusalem; praise thy God, ...","praise lord , jerusalem ; praise thy god , zio..."
298,298,Orthodox,Psalter,148,Praise ye the Lord from the heavens; praise Hi...,praise ye lord heaven ; praise highest . prais...
299,299,Orthodox,Psalter,149,"Sing unto the Lord a new song, His praise is i...","sing unto lord new song , praise congregation ..."


# Converting Data to `txt` files. 

Based on the Github repo, we need to do the training on a single txt file. I am considering each psalm to be a single document. Therefore we need to take the column of **cleaned_verse** and combined them into a single tx file. Since adding the label of each document would get in the way, I am making 2 parallel files

**Corpus for GloVe - `corpus.txt`**
1. Blessed is the man who walks not in the counsel of the ungodly...
2. Why do the nations rage, and the people plot in vain...
3. Blessed is the man that walketh not in the counsel of the ungodly...
4. Why do the heathen rage, and the people imagine a vain thing...

**Psalm Index - `corpus_index.txt`**

| Line | Psalm   | Tradition |
|------|---------|-----------|
| 1    | Psalm 1 | Psalter   |
| 2    | Psalm 2 | Psalter   |
| 3    | Psalm 1 | Bible     |
| 4    | Psalm 2 | Bible     |



In [6]:
with open("corpus.txt", "w", encoding="utf-8") as corpus_file, \
     open("corpus_index.txt", "w", encoding="utf-8") as index_file:
    
    for line_number, row in enumerate(psalms.itertuples(index=False), start=1):
        # 1. Corpus: cleaned text, one Psalm per line
        corpus_file.write(str(row.cleaned_psalm).strip().replace("\n", " ") + "\n")
    
        
        # 2. Index file: line number → Psalm ## + tradition
        index_file.write(f"{line_number}\tPsalm {row.psalm_num}\t{row.text}\n")

# Generating GloVe Models

After reading through the Github and using a bit of ChatGPT, I was able to compile,m som functions to generate the `GloVe` components needed. This is the same thing as using the command line arguments via the terminal. Lets test them out. 


In [7]:
import glove_utils as gu

# Make sure corpus.txt already exists and contains your training text

vectors_file = gu.train_glove("corpus.txt", glove_path="./glove")


BUILDING VOCABULARY
Processed 0 tokens.Processed 50280 tokens.
Counted 3473 unique words.
Using vocabulary of size 3473.

COUNTING COOCCURRENCES
window size: 10
context: symmetric
max product: 13752509
overflow length: 38028356
Reading vocab from file "vocab.txt"...loaded 3473 words.
Building lookup table...table contains 12061730 elements.
Processing token: 0Processed 50280 tokens.
Writing cooccurrences to disk.......2 files in total.
Merging cooccurrence files: processed 0 lines.0 lines.100000 lines.200000 lines.300000 lines.Merging cooccurrence files: processed 311775 lines.

Using random seed 1758555313
SHUFFLING COOCCURRENCES
array size: 255013683
Shuffling by chunks: processed 0 lines.processed 311775 lines.
Wrote 1 temporary file(s).
Merging temp files: processed 0 lines.311775 lines.Merging temp files: processed 311775 lines.

TRAINING MODEL
Read 311775 lines.
Initializing parameters...Using random seed 1758555314
done.
vector size: 100
vocab size: 3473
x_max: 10.000000
alpha: 

Vectors saved to vectors.txt


In [8]:
# Generating vectors
glove_vectors = gu.load_glove(vectors_file)

In [14]:
# Testing 
# User input
query = input("Enter something to search for: ")
print(glove_vectors.get(query))


[ 0.589839 -0.311932  0.298654 -0.004388 -0.024171 -0.504709 -0.28803
  0.474389  0.236727  0.197487  0.44909   0.102896  0.368502  0.04483
  0.106567  0.070447  0.061163  0.201818  0.242514  0.054805 -0.637505
  0.058717  0.231961  0.488454 -0.197468  0.033281 -0.352794 -0.359831
 -0.164271  0.196394 -0.105344  0.46006  -0.063066 -0.484431 -0.121765
  0.216275 -0.094139 -0.722933  0.098848 -0.457943  0.075742  0.077969
 -0.311726  0.146272  0.168047 -0.086328 -0.186107 -0.053884 -0.342455
  0.280366  0.262745  0.25028   0.212974  0.283869 -0.029683  0.12536
 -0.085864 -0.163667 -0.380764 -0.023007  0.230003  0.21496  -0.029982
 -0.200549 -0.061691  0.057967  0.078204  0.320915 -0.335692 -0.126157
  0.237741  0.243009 -0.029097  0.087554 -0.133909  0.077971 -0.337901
 -0.248183  0.297987 -0.072876  0.096889  0.073372  0.523179 -0.133393
  0.212228 -0.004691  0.087707 -0.149595 -0.028021 -0.213859  0.32915
  0.088136 -0.330031 -0.259912  0.075179 -0.025214 -0.184754 -0.001805
  0.411027

#
With the GloVe Model trained, let's prototype the the searcn results just like the `TF-IDF` results.  

In [15]:
def psalm_embedding(text):
    words = text.lower().split()  # simple tokenization
    vectors = [glove_vectors[w] for w in words if w in glove_vectors]
    if len(vectors) == 0:
        return np.zeros(next(iter(glove_vectors.values())).shape)
    return np.mean(vectors, axis=0)

# Add a column with embeddings
psalms['glove_vec'] = psalms['cleaned_psalm'].apply(psalm_embedding)


In [16]:
psalms

,Unnamed: 0,tradition,text,psalm_num,psalm,cleaned_psalm,glove_vec
0,0,Orthodox,Bible,1,Blessed is the man Who walks not in the counse...,blessed man walk counsel ungodly stand way sin...,"[-0.2277687, -0.08095848, -0.098934636, 0.1080..."
1,1,Orthodox,Bible,2,Why do the nations rage And the people meditat...,nation rage people meditate vain thing king ea...,"[-0.20237091, -0.24262428, -0.093284406, 0.213..."
2,2,Orthodox,Bible,3,A psalm by David when he fled from the face of...,psalm david fled face son absalom olord afflic...,"[-0.30312675, -0.11780859, -0.19432178, 0.1651..."
3,3,Orthodox,Bible,4,For the End in psalms an ode by David You hear...,end psalm ode david heard icalled god righteou...,"[-0.2721842, -0.11145677, -0.054780994, 0.2410..."
4,4,Orthodox,Bible,5,For the End concerning the inheritance a psalm...,end concerning inheritance psalm david give ea...,"[-0.23068598, -0.21158625, -0.034757618, 0.214..."
...,...,...,...,...,...,...,...
296,296,Orthodox,Psalter,146,The Lord doth build up Jerusalem; He shall gat...,lord doth build jerusalem ; shall gather toget...,"[-0.26162377, -0.30188063, 0.025113385, 0.2583..."
297,297,Orthodox,Psalter,147,"Praise the Lord, O Jerusalem; praise thy God, ...","praise lord , jerusalem ; praise thy god , zio...","[-0.30648193, -0.5041665, 0.013473133, 0.41855..."
298,298,Orthodox,Psalter,148,Praise ye the Lord from the heavens; praise Hi...,praise ye lord heaven ; praise highest . prais...,"[-0.38472998, -0.48913506, 0.14994206, 0.54637..."
299,299,Orthodox,Psalter,149,"Sing unto the Lord a new song, His praise is i...","sing unto lord new song , praise congregation ...","[-0.29447573, -0.45995724, 0.07865666, 0.33374..."


In [ ]:
def query_glove(query):
    q_vec = psalm_embedding(query)
    
    results = []
    for _, row in psalms.iterrows():
        sim = np.dot(q_vec, row['glove_vec']) / (np.linalg.norm(q_vec) * np.linalg.norm(row['glove_vec']))
        results.append({
            "text": row['text'],
            "psalm_num": row['psalm_num'],
            "psalm": row['psalm'],
            "similarity": round(sim*100, 2)
        })
    
    # Sort by similarity
    results.sort(key=lambda x: x['similarity'], reverse=True)
    
    return results[:6]  # top 6 results

In [26]:
query = input("Enter a query to search: ")
results = query_glove(query)

results

UFuncTypeError: ufunc 'add' did not contain a loop with signature matching types (dtype('float64'), dtype('<U1')) -> None